# ML-07 - Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faisaljaam002-png/flyrank-assignment1/blob/main/work/notebooks/w04_baseline_score.ipynb)

**Lane 2 - Refresh / Content Opportunity Scoring.** One row = one content page. The output is a
ranked editor queue: which page should a human fix **first**?

This notebook does three things: (1) audit the two signals my rule leans on, (2) encode ONE
transparent rule with a score, one reason code, and one action label, and (3) read my own
top-ten with a skeptic's eye. The queue is written to `work/outputs/baseline_action_score.csv`,
and a metrics receipt to `work/outputs/w04_baseline_metrics.json`.

Skill: `building-baselines` + `flyrank/flyrank-data` (loaded from `skills/README.md`).

## 0. Setup (Colab or local)

On Colab this clones the repo and installs requirements. Locally it just moves to the repo root and loads the starter slice.

In [1]:
import os, sys, json, subprocess
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faisaljaam002-png/flyrank-assignment1"

if IN_COLAB:
    if not os.path.isdir("flyrank-assignment1"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-assignment1"], check=True)
    os.chdir("flyrank-assignment1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")
elif os.path.basename(os.getcwd()) == "work":
    os.chdir("..")

print("Working dir:", os.getcwd())
assert Path("data/raw/content_refresh_anonymized.csv").exists(), "starter CSV not found"
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# Label is used ONLY for evaluation display. It is never an input to the rule.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["trend_direction"] = df["trend_direction"].astype(str)
print(f"Loaded {len(df):,} pages, {df['client_id'].nunique()} clients, {df.shape[1]} columns")
print(f"Label positive rate (declining): {df['is_declining_label'].mean():.3f}")

Working dir: E:\FlyRank Ai\flyrank-assignment1


Loaded 30,000 pages, 32 clients, 45 columns
Label positive rate (declining): 0.542


## 1. My rule and its reason codes

### The rule, in plain words

> **A page is worth an editor's review when it sits on page 1-2 where clicks should be flowing,
> but its click-through rate is far below what pages at that position typically earn. The larger
> the gap between its actual CTR and the field, the higher the priority. A page left un-updated
> for a long time (104+ days) gets a modest bump. When urgency ties, the page with more traffic
> at stake ranks first.**

Why 104+ days? The staleness audit below shows freshness is only informative at the far tail -
moderate staleness is not a decline signal. Gating staleness at 104 days is the audit's direct
edit to my rule.

### The two signals the rule leans on (checked first, per the assignment)

Both signals are behind real FlyRank flags from the session:
**staleness** is the assumption behind the refresh flags, and **CTR-vs-position** is the
assumption behind the CTR-fix logic. Each test below prints a bucket table with n.

### Signal check #1 - CTR-vs-position (behind the CTR-fix logic)

**Claim being tested:** *among pages that reach page 1-2 (position 1-20), the ones with lower
CTR are more likely to be in decline.* The CTR-fix flag only makes sense if a good position with
a bad click rate marks a page that is losing value.

Bucket table: position 1-20, split into CTR bands, n printed.

In [2]:
pos12 = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
ctr_band = pd.cut(pos12["ctr"], [0, 0.1, 0.5, 1.5, 100], labels=["<0.1", "0.1-0.5", "0.5-1.5", ">1.5"])
t = pos12.groupby(ctr_band, observed=True)["is_declining_label"].agg(n="size", declining_rate="mean").round(4)
t["n_pct"] = (t["n"] / len(pos12) * 100).round(1)
print(f"Pages on position 1-20: n = {len(pos12):,}   (overall declining rate {pos12['is_declining_label'].mean():.3f})")
print(t)
# focus on the exact flag population: visible zero-click pages on page 1-2
z = (df["impressions_90d"] >= 300) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] == 0)
print(f"Visible zero-CTR pages on page 1-2: n = {z.sum():,}, declining rate {df.loc[z, 'is_declining_label'].mean():.3f}")
print("VERDICT: CONFIRMED - decline rate falls monotonically from 0.67 (CTR<0.1) to 0.42 (CTR>1.5).")

Pages on position 1-20: n = 20,256   (overall declining rate 0.580)
            n  declining_rate  n_pct
ctr                                 
<0.1     1916          0.6717    9.5
0.1-0.5  7233          0.5941   35.7
0.5-1.5  2699          0.5091   13.3
>1.5      920          0.4185    4.5
Visible zero-CTR pages on page 1-2: n = 1,826, declining rate 0.762
VERDICT: CONFIRMED - decline rate falls monotonically from 0.67 (CTR<0.1) to 0.42 (CTR>1.5).


### Signal check #2 - staleness (behind the refresh flags)

**Claim being tested:** *pages that have not been updated in a long time are more likely to be
declining.* The refresh flags assume old = at risk.

Bucket table: visible pages (at least 300 impressions) split by days-since-update, n printed.

In [3]:
vis = df[df["impressions_90d"] >= 300].copy()
bins = pd.cut(vis["days_since_last_update"], [0, 30, 90, 103, 179, 10**6],
              labels=["0-30", "31-90", "91-103", "104-179", "180+"])
t = vis.groupby(bins, observed=True)["is_declining_label"].agg(n="size", declining_rate="mean").round(4)
print(f"Visible pages (impressions >= 300): n = {len(vis):,}   (overall declining rate {vis['is_declining_label'].mean():.3f})")
print(t)
print("All pages, 180+ days untouched:",
      f"n = {(df['days_since_last_update'] >= 180).sum():,}, "
      f"declining rate {df.loc[df['days_since_last_update'] >= 180, 'is_declining_label'].mean():.3f}")
print("VERDICT: MIXED - the relationship is not monotonic. Only the far tail (104+ days) is elevated;")
print("freshly-refreshed pages do not decline less across the whole range. The rule gates staleness at 104+ days only.")

Visible pages (impressions >= 300): n = 18,752   (overall declining rate 0.595)
                            n  declining_rate
days_since_last_update                       
0-30                    11398          0.5810
31-90                     120          0.5583
91-103                     40          0.2500
104-179                  7172          0.6189
180+                       22          0.8182
All pages, 180+ days untouched: n = 174, declining rate 0.471
VERDICT: MIXED - the relationship is not monotonic. Only the far tail (104+ days) is elevated;
freshly-refreshed pages do not decline less across the whole range. The rule gates staleness at 104+ days only.


### What the audits changed in my rule

- **CTR-vs-position (CONFIRMED)** gets the most weight (0.85): it is the cleanest, most
  monotonic signal I found.
- **Staleness (MIXED)** was demoted to a gated bonus (0.10) that only fires at the far tail
  (104+ days). Moderate staleness is not scored - the audit saved the rule from over-weighting a
  weak signal.
- **Volume** is *not* a decline predictor in this data (high-volume pages decline at or below
  the base rate), so it enters only as a 0.15 tie-breaker - the size of the prize, never a risk
  flag. This is the "volume behind quick-win" flag used honestly: demand sizes the opportunity;
  it does not predict decline.

### Reason codes (exactly one per row) and action labels

| Reason code | Fires when | Action label |
|---|---|---|
| `ctr_lag_visible` | position 1-20, impressions >= 300, CTR < 0.5 | `refresh_and_review_ctr` |
| `stale_visible` | position 1-20, impressions >= 300, >=104 days untouched, CTR >= 0.5 | `refresh` |
| `visible_opportunity` | impressions >= 300, no urgent flag above | `watch_and_monitor` |
| `low_traffic_monitor` | impressions < 300 | `monitor` |

## 2. Build the ranked queue (writes the CSV)

In [4]:
def percentile_rank(s):
    return pd.to_numeric(s, errors="coerce").fillna(0).rank(method="average", pct=True).fillna(0)

on_page12   = ((df["avg_position"] > 0) & (df["avg_position"] <= 20))
ctr_gap     = on_page12 * (1 - percentile_rank(df["ctr"]))          # how far below the field CTR sits
stale_tail  = (on_page12 & (df["days_since_last_update"] >= 104)).astype(int)  # far-tail staleness (audited)
volume_pct  = percentile_rank(np.log1p(df["impressions_90d"]))      # size of the prize (tie-breaker)

df["score"] = (0.85 * ctr_gap + 0.10 * stale_tail + 0.15 * volume_pct)

visible     = df["impressions_90d"] >= 300
low_ctr     = on_page12 & visible & (df["ctr"] < 0.5)
stale_vis   = on_page12 & visible & (df["days_since_last_update"] >= 104)
df["reason_code"] = np.select(
    [low_ctr, stale_vis, visible],
    ["ctr_lag_visible", "stale_visible", "visible_opportunity"],
    default="low_traffic_monitor")
df["action_label"] = np.select(
    [low_ctr, stale_vis, visible],
    ["refresh_and_review_ctr", "refresh", "watch_and_monitor"],
    default="monitor")

queue = df.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = queue.index + 1

out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
cols = ["rank", "content_id", "client_id", "score", "reason_code", "action_label",
        "impressions_90d", "clicks_90d", "ctr", "avg_position",
        "days_since_last_update", "content_age_days",
        "is_declining_label"]  # label is eval-only display, never a rule input
queue[cols].to_csv(out_path, index=False)
print("Wrote", out_path)

# Evaluation (label used only here, to be honest about how the queue performs)
base_rate = queue["is_declining_label"].mean()
def prec_at_k(k):
    return queue.head(k)["is_declining_label"].mean()
print(f"Queue rows: {len(queue):,}")
print(f"Base declining rate: {base_rate:.3f}")
print(f"Precision@10 = {prec_at_k(10):.3f}  |  Precision@50 = {prec_at_k(50):.3f}")
print("Reason code counts:")
print(queue["reason_code"].value_counts().to_string())
print("Action label counts (top 50):")
print(queue.head(50)["action_label"].value_counts().to_string())

metrics = {
    "rows": int(len(queue)),
    "base_declining_rate": round(float(base_rate), 4),
    "precision_at_10": round(float(prec_at_k(10)), 4),
    "precision_at_50": round(float(prec_at_k(50)), 4),
    "score_formula": {"ctr_gap": 0.85, "stale_tail_gated_104d": 0.10, "volume_tiebreak": 0.15},
    "verdicts": {"ctr_vs_position_ctr_fix_flag": "CONFIRMED", "staleness_refresh_flags": "MIXED"},
    "reason_code_counts": queue["reason_code"].value_counts().to_dict(),
    "csv_output": "work/outputs/baseline_action_score.csv",
}
with open("work/outputs/w04_baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, sort_keys=True)
print("Wrote work/outputs/w04_baseline_metrics.json")

Wrote work\outputs\baseline_action_score.csv
Queue rows: 30,000
Base declining rate: 0.542
Precision@10 = 0.800  |  Precision@50 = 0.800
Reason code counts:
reason_code
low_traffic_monitor    11248
ctr_lag_visible        10730
visible_opportunity     7159
stale_visible            863
Action label counts (top 50):
action_label
refresh_and_review_ctr    50
Wrote work/outputs/w04_baseline_metrics.json


## 3. Top-10 review (a skeptic's eye)

For each of the top ten: the action, **why it is there**, and **what would make it wrong**.
Ranks 11-20 follow as a compact table (top-20 is a welcome bonus; ten is the requirement).

In [5]:
review = {
 1: ("refresh_and_review_ctr", "208k impressions, position 9.7, ZERO clicks in 90 days - the single biggest silent CTR failure in the slice; it is also already declining.", "If the impressions are navigational/branded queries where zero clicks is expected, or if ctr=0 is a tracking gap rather than reality."),
 2: ("refresh_and_review_ctr", "16.8k impressions at position 5.6, zero clicks, declining. Top-of-page-1 exposure with no click-through.", "If its zero-CTR is driven by one or two non-search bots inflating impressions."),
 3: ("refresh_and_review_ctr", "16.2k impressions at position 9.0, zero clicks, declining. Page-1 slot wasting its exposure.", "If the position is recent (page just climbed) and the 90-day CTR averages over an older, lower-traffic period."),
 4: ("refresh_and_review_ctr", "7.7k impressions at position 8.3, zero clicks, declining. Same zero-click pattern, smaller prize.", "If impressions are mostly from its own client's internal navigation."),
 5: ("refresh_and_review_ctr", "7.1k impressions at position 16, zero clicks, but trend is STABLE. High urgency by my rule, no decline label.", "The rule says fix it; the stable trend says maybe the zero-CTR is a stable quirk. Manual look before a refresh decision."),
 6: ("refresh_and_review_ctr", "6.8k impressions at position 16.1, zero clicks, declining. Lower page-1/2 band, still zero clicks.", "If a snippet or title issue is unfixable within an edit budget - then it is a monitoring case, not a refresh case."),
 7: ("refresh_and_review_ctr", "6.6k impressions at position 5.2, zero clicks, declining. High position, no clicks.", "If it is a compare/listing page whose impressions are cheap position hits with no intent to click."),
 8: ("refresh_and_review_ctr", "6.5k impressions at position 17.4, zero clicks, declining. Untouched 106 days.", "If the page's target query has seasonally collapsed and the decline is demand-driven, not click-fixable."),
 9: ("refresh_and_review_ctr", "6.5k impressions at position 9.3, zero clicks, but trend STABLE. Rule ranks it high; the label disagrees.", "The stable label is a check: this may be a page whose zero CTR is 'normal' for its query. Verify before editing."),
10: ("refresh_and_review_ctr", "5.7k impressions at position 8.3, zero clicks, declining. Page-1 slot with no clicks for a full quarter.", "If a tracking change mid-window reset click counting - then the flag is an artifact."),
}
for r, (action, why, wrong) in review.items():
    row = queue.iloc[r - 1]
    print(f"Rank {r:>2} | {action} | {row['content_id']} | score {row['score']:.3f} | imp {int(row['impressions_90d']):,} | pos {row['avg_position']} | ctr {row['ctr']}")
    print(f"        why: {why}")
    print(f"        would be wrong if: {wrong}")
print()
print("Ranks 11-20 (same action; declining rate 0.70):")
print(queue.iloc[10:20][["rank", "content_id", "score", "impressions_90d", "ctr", "avg_position", "is_declining_label"]].round(3).to_string(index=False))

Rank  1 | refresh_and_review_ctr | content_c8e9d6ab9013 | score 0.913 | imp 208,678 | pos 9.7 | ctr 0.0
        why: 208k impressions, position 9.7, ZERO clicks in 90 days - the single biggest silent CTR failure in the slice; it is also already declining.
        would be wrong if: If the impressions are navigational/branded queries where zero clicks is expected, or if ctr=0 is a tracking gap rather than reality.
Rank  2 | refresh_and_review_ctr | content_825a9788af8d | score 0.902 | imp 16,786 | pos 5.6 | ctr 0.0
        why: 16.8k impressions at position 5.6, zero clicks, declining. Top-of-page-1 exposure with no click-through.
        would be wrong if: If its zero-CTR is driven by one or two non-search bots inflating impressions.
Rank  3 | refresh_and_review_ctr | content_8ba781dafa55 | score 0.901 | imp 16,156 | pos 9.0 | ctr 0.0
        why: 16.2k impressions at position 9.0, zero clicks, declining. Page-1 slot wasting its exposure.
        would be wrong if: If the position is r

## 4. Weak picks + leakage check

**Weak picks in my own top-20:** ranks 5, 9, and 15-16 are flagged `refresh_and_review_ctr` yet
their label is *stable* or *up*. My rule treats zero clicks on page 1-2 as always-urgent; the
stable/up labels show that is not always true - some queries simply never click, and the top of
the list is probably over-flagging a few of them. That is the biggest known failure mode and the
model must do better on it.

**Leakage check:** the rule inputs are `avg_position`, `ctr`, `impressions_90d`,
`days_since_last_update`. All are trailing-90-day values knowable at export time. The rule never
reads `trend_direction`, `trend_pct`, the `*_last_30d` / `*_prev_30d` windows, or any product
flag. The label is appended to the CSV for evaluation display only.

In [6]:
# Which of my top-20 are NOT declining? (the weak picks)
top20 = queue.head(20)
weak = top20[top20["is_declining_label"] == 0]
print("Top-20 not-declining (potential over-flags):", len(weak), "of 20")
for _, row in weak.iterrows():
    print(f"  rank {int(row['rank']):>2} | {row['reason_code']} | trend={row['trend_direction']} | imp {int(row['impressions_90d']):,}")

# Leakage guard: the rule's inputs must never touch label-derived or future-window columns.
RULE_INPUTS = ["avg_position", "ctr", "impressions_90d", "days_since_last_update"]
LEAKY = ["trend_direction", "trend_pct", "is_declining_label",
         "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
         "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
assert not (set(RULE_INPUTS) & set(LEAKY)), "a rule input is label-derived!"
print("Rule inputs:", RULE_INPUTS)
print("No leaky column is used by the rule. OK.")
print("CSV label column is evaluation-only: it is not read when scores are computed.")

Top-20 not-declining (potential over-flags): 5 of 20
  rank  5 | ctr_lag_visible | trend=stable | imp 7,087
  rank  9 | ctr_lag_visible | trend=stable | imp 6,524
  rank 15 | ctr_lag_visible | trend=stable | imp 4,446
  rank 16 | ctr_lag_visible | trend=stable | imp 4,230
  rank 20 | ctr_lag_visible | trend=up | imp 3,881
Rule inputs: ['avg_position', 'ctr', 'impressions_90d', 'days_since_last_update']
No leaky column is used by the rule. OK.
CSV label column is evaluation-only: it is not read when scores are computed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two signal verdicts with bucket tables and n (both flag-linked: CTR-fix and refresh flags)
- [x] One rule: a score, exactly one reason code per row, and an action label
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`
- [x] Ten reviewed rows, each with "what would make it wrong"
- [x] No future-window or label-derived inputs in the rule
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.